In [6]:
import os
os.chdir(r'C:\Users\pc\OneDrive\Desktop\ecommerce-analytics\notebooks')
print("Working directory:", os.getcwd())

Working directory: C:\Users\pc\OneDrive\Desktop\ecommerce-analytics\notebooks


In [7]:
import duckdb
import pandas as pd
import numpy as np

DATA_PATH = r'C:\Users\pc\OneDrive\Desktop\ecommerce-analytics\data'
DB_PATH   = r'C:\Users\pc\OneDrive\Desktop\ecommerce-analytics\data\ecommerce.db'

# Load all 9 files
files = {
    'orders'      : 'olist_orders_dataset.csv',
    'items'       : 'olist_order_items_dataset.csv',
    'payments'    : 'olist_order_payments_dataset.csv',
    'reviews'     : 'olist_order_reviews_dataset.csv',
    'customers'   : 'olist_customers_dataset.csv',
    'products'    : 'olist_products_dataset.csv',
    'sellers'     : 'olist_sellers_dataset.csv',
    'category'    : 'product_category_name_translation.csv',
    'geolocation' : 'olist_geolocation_dataset.csv'
}

dfs = {}
for name, filename in files.items():
    path      = os.path.join(DATA_PATH, filename)
    dfs[name] = pd.read_csv(path)
    print(f"✅ {name:<15} {dfs[name].shape[0]:>10,} rows")

✅ orders              99,441 rows
✅ items              112,650 rows
✅ payments           103,886 rows
✅ reviews             99,224 rows
✅ customers           99,441 rows
✅ products            32,951 rows
✅ sellers              3,095 rows
✅ category                71 rows
✅ geolocation      1,000,163 rows


In [8]:
# Convert all date columns
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    dfs['orders'][col] = pd.to_datetime(
        dfs['orders'][col], errors='coerce'
    ).astype(str)  # store as string for DuckDB compatibility

# Clean reviews — fill missing review comments
dfs['reviews']['review_comment_title']   = dfs['reviews']['review_comment_title'].fillna('')
dfs['reviews']['review_comment_message'] = dfs['reviews']['review_comment_message'].fillna('')

# Clean products — fill missing category
dfs['products']['product_category_name'] = dfs['products']['product_category_name'].fillna('unknown')

print("✅ Data cleaning complete")
print(f"\nOrders date range:")
print(f"  Min: {dfs['orders']['order_purchase_timestamp'].min()}")
print(f"  Max: {dfs['orders']['order_purchase_timestamp'].max()}")

✅ Data cleaning complete

Orders date range:
  Min: 2016-09-04 21:15:19
  Max: 2018-10-17 17:30:18


In [13]:
import os
con = duckdb.connect(DB_PATH)

for table_name, df in table_map.items():
    # Save to parquet first then load — bypasses type issues completely
    parquet_path = os.path.join(DATA_PATH, f'{table_name}.parquet')
    df.to_parquet(parquet_path, index=False)
    
    con.execute(f"DROP TABLE IF EXISTS {table_name}")
    con.execute(f"""
        CREATE TABLE {table_name} AS 
        SELECT * FROM read_parquet('{parquet_path.replace(chr(92), '/')}')
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"✅ {table_name:<25} {count:>10,} rows")

con.close()
print("\n✅ All 9 tables loaded into ecommerce.db")

✅ raw_orders                    99,441 rows
✅ raw_items                    112,650 rows
✅ raw_payments                 103,886 rows
✅ raw_reviews                   99,224 rows
✅ raw_customers                 99,441 rows
✅ raw_products                  32,951 rows
✅ raw_sellers                    3,095 rows
✅ raw_category                      71 rows
✅ raw_geolocation            1,000,163 rows

✅ All 9 tables loaded into ecommerce.db


In [19]:
con = duckdb.connect(DB_PATH)

# Check all tables across all schemas
print(con.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").df())

con.close()

  table_schema       table_name
0         main     raw_category
1         main    raw_customers
2         main  raw_geolocation
3         main        raw_items
4         main     raw_payments
5         main     raw_products
6         main      raw_reviews
7         main      raw_sellers


In [20]:
con = duckdb.connect(DB_PATH)

# Reload orders specifically
df_orders = dfs['orders'].copy()
for col in df_orders.select_dtypes(include=['object']).columns:
    df_orders[col] = df_orders[col].astype(str)

parquet_path = os.path.join(DATA_PATH, 'raw_orders.parquet').replace('\\', '/')

df_orders.to_parquet(parquet_path, index=False)

con.execute("DROP TABLE IF EXISTS raw_orders")
con.execute(f"CREATE TABLE raw_orders AS SELECT * FROM read_parquet('{parquet_path}')")

count = con.execute("SELECT COUNT(*) FROM raw_orders").fetchone()[0]
print(f"✅ raw_orders loaded with {count:,} rows")

# Verify all 9 tables now exist
print("\n=== ALL TABLES ===")
print(con.execute("SHOW TABLES").df())

con.close()

C:\Users\pc\AppData\Local\Temp\ipykernel_2664\1487581026.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_orders.select_dtypes(include=['object']).columns:


✅ raw_orders loaded with 99,441 rows

=== ALL TABLES ===
              name
0     raw_category
1    raw_customers
2  raw_geolocation
3        raw_items
4       raw_orders
5     raw_payments
6     raw_products
7      raw_reviews
8      raw_sellers


In [21]:
con = duckdb.connect(DB_PATH)

print("=== ORDER STATUS DISTRIBUTION ===")
print(con.execute("""
    SELECT order_status,
           COUNT(*)                                   AS count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*))
                 OVER(), 2)                           AS pct
    FROM raw_orders
    GROUP BY order_status
    ORDER BY count DESC
""").df())

print("\n=== REVENUE SUMMARY ===")
print(con.execute("""
    SELECT
        COUNT(DISTINCT o.order_id)         AS total_orders,
        ROUND(SUM(p.payment_value), 2)     AS total_revenue,
        ROUND(AVG(p.payment_value), 2)     AS avg_order_value,
        COUNT(DISTINCT o.customer_id)      AS unique_customers
    FROM raw_orders o
    JOIN raw_payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
""").df())

con.close()

=== ORDER STATUS DISTRIBUTION ===
  order_status  count    pct
0    delivered  96478  97.02
1      shipped   1107   1.11
2     canceled    625   0.63
3  unavailable    609   0.61
4     invoiced    314   0.32
5   processing    301   0.30
6      created      5   0.01
7     approved      2   0.00

=== REVENUE SUMMARY ===
   total_orders  total_revenue  avg_order_value  unique_customers
0         96477    15422461.77           153.07             96477


In [1]:
import duckdb

DB_PATH = r'C:\Users\pc\OneDrive\Desktop\ecommerce-analytics\data\ecommerce.db'
con = duckdb.connect(DB_PATH)

print("=== ALL TABLES ===")
print(con.execute("SHOW TABLES").df())

print("\n=== EXECUTIVE SUMMARY ===")
print(con.execute("SELECT * FROM executive_summary").df())

print("\n=== RFM SEGMENTS ===")
print(con.execute("""
    SELECT rfm_segment,
           COUNT(*)                        AS customers,
           ROUND(AVG(monetary), 2)         AS avg_spend,
           ROUND(AVG(frequency), 2)        AS avg_orders,
           ROUND(AVG(recency_days), 0)     AS avg_recency_days
    FROM customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_spend DESC
""").df())

print("\n=== TOP 5 CATEGORIES ===")
print(con.execute("""
    SELECT category,
           SUM(total_orders)               AS orders,
           ROUND(SUM(total_revenue), 2)    AS revenue,
           ROUND(AVG(avg_rating), 2)       AS avg_rating
    FROM product_performance
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 5
""").df())

con.close()

=== ALL TABLES ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                   name
0          customer_rfm
1     executive_summary
2   product_performance
3          raw_category
4         raw_customers
5       raw_geolocation
6             raw_items
7            raw_orders
8          raw_payments
9          raw_products
10          raw_reviews
11          raw_sellers
12    stg_order_details
13           stg_orders
14         stg_products

=== EXECUTIVE SUMMARY ===
   total_orders  total_customers  total_revenue  avg_order_value  \
0         96478            93358    19776160.44           179.47   

   max_order_value  avg_review_score  avg_delivery_days  late_delivery_pct  \
0         13664.08              4.08               12.4               7.91   

     first_order_date     last_order_date  
0 2016-09-15 12:16:38 2018-08-29 15:00:37  

=== RFM SEGMENTS ===
           rfm_segment  customers  avg_spend  avg_orders  avg_recency_days
0       Cant Lose Them      13250     306.32        1.00             500.0
1  Potential Loyalists       5071  